In [ ]:
// 匹配所有节点，返回所有节点（无LIMIT限制，不折叠）
MATCH (n) RETURN n;

In [1]:
import pandas as pd
import re
from neo4j import GraphDatabase

# ==========================================
# --- 1. 核心配置与全局变量 ---
# ==========================================

# 气体基础自由能配置
GAS_G = {
    'H2': -6.735,      # 对应 1(H+ + e-) = 0.5 * G(H2)
    '*': -559.364,     # 清洁表面(Clean Surface)的自由能
}

# ==========================================
# --- 2. 核心处理函数 ---
# ==========================================

def calculate_step_dg(equation, energy_dict):
    """
    解析方程式并计算 free_energy，支持带化学计量数的物种 (例如 2*, 2H+, *CO2)
    """
    try:
        reactants_part, products_part = equation.split('->')
    except:
        return None

    def get_part_energy(part_str):
        total_g = 0
        
        # 匹配 "可选的数字系数" + "物种"
        pattern = r'(\d+(?:\.\d+)?)?\s*(\*[A-Za-z0-9]*|[A-Za-z0-9]+[\+\-]?)'
        matches = re.findall(pattern, part_str)
        
        for coeff_str, species in matches:
            if not species: 
                continue
                
            coeff = float(coeff_str) if coeff_str else 1.0
            
            # 1. 处理质子 H+ (CHE模型)
            if species == 'H+':
                total_g += coeff * 0.5 * energy_dict.get('H2', 0)
                continue
            
            # 2. 处理电子 e- (已在H+中处理)
            if species in ['e-', 'e']:
                continue
            
            # 3. 处理其他化学物种
            if species in energy_dict:
                total_g += coeff * energy_dict[species]
            else:
                print(f"警告: 找不到物种 [{species}] 的能量数据，方程式为: {part_str.strip()}")
        
        return total_g

    g_react = get_part_energy(reactants_part)
    g_prod = get_part_energy(products_part)
    
    return g_prod - g_react


def clean_and_split(text):
    """将 A + B 格式拆分为列表，并过滤掉不需要的干扰项"""
    # 排除项：水、质子、电子以及孤立单星号位 (这里按原代码逻辑保留了 '*' 被过滤，如需修改请调整)
    exclude = {'H2O', '*H2O', 'H+', 'e-', 'H', 'e', '*'}
    
    species = [s.strip() for s in text.split("+") if s.strip()]
    return [s for s in species if s not in exclude]


# ==========================================
# --- 3. 数据处理与
# ==========================================

def process_data_and_run_neo4j(cal_csv, node_edge_csv, output_csv):
    print(">>> [1/4] 读取数据...")
    energy_df = pd.read_csv(cal_csv) 
    df = pd.read_csv(node_edge_csv, encoding='utf-8')

    # 将能量表转为字典并合并 GAS_G
    energy_dict = dict(zip(energy_df['id'], energy_df['Gibbs free energy at 298.1 K (eV)']))
    energy_dict.update(GAS_G)

    print(">>> [2/4] 清洗方程式并计算 free_energy...")
    # 统一箭头格式
    df['response equation'] = df['response equation'].astype(str) \
        .str.replace('鈫?', '->', regex=False) \
        .str.replace('→', '->', regex=False)

    # 计算自由能并存储
    df['free_energy'] = df['response equation'].apply(
        lambda eq: round(calculate_step_dg(eq, energy_dict), 4) if calculate_step_dg(eq, energy_dict) is not None else "Error"
    )

    print(">>> [3/4] 识别物种位置并进行矩阵编码...")
    all_species_set = set()
    for eqn in df['response equation']:
        if '->' not in eqn: continue
        left_part, right_part = eqn.split('->', maxsplit=1)
        all_species_set.update(clean_and_split(left_part))
        all_species_set.update(clean_and_split(right_part))

    unique_species = sorted(list(all_species_set))

    # 初始化物种列和其他辅助列
    for spec in unique_species:
        df[spec] = 0
    df['reaction_id'] = ""
    df['reactant_combination'] = ""

    # 根据位置填充编码
    for index, row in df.iterrows():
        eqn = str(row['response equation']).strip()
        if '->' not in eqn:
            df.loc[index, 'reactant_combination'] = "无效方程"
            continue

        left_side, right_side = eqn.split('->', maxsplit=1)
        reactants = clean_and_split(left_side)
        products = clean_and_split(right_side)

        df.loc[index, 'reaction_id'] = f"REACT_{str(index + 1).zfill(3)}"
        df.loc[index, 'reactant_combination'] = "+".join(sorted(reactants))

        # 规则 1：反应物，编码 -1
        for s in reactants:
            if s in unique_species:
                df.loc[index, s] = -1

        # 规则 2：生成物
        for s in products:
            if s in unique_species:
                if s.startswith("*"):
                    df.loc[index, s] = 1  # 吸附态
                else:
                    df.loc[index, s] = 2  # 气相

    # 重组列顺序：基本信息 -> free_energy -> 矩阵编码
    meta_cols = ['reaction_id', 'reactant_combination', 'response equation', 'free_energy']
    final_cols = meta_cols + unique_species
    df = df[final_cols]
    
    df.to_csv(output_csv, index=False, encoding='utf-8')
    print(f"数据处理完成！已保存至 {output_csv}。")



if __name__ == "__main__":
    # 配置你的文件路径
    CAL_DATA_CSV = "cal_data.csv"
    NODE_EDGE_CSV = "node_edge.csv"
    OUTPUT_CSV = "processed_reaction_results.csv"  # 我将输出文件统一换了一个新名字防止覆盖源文件
    
    process_data_and_run_neo4j(CAL_DATA_CSV, NODE_EDGE_CSV, OUTPUT_CSV)

>>> [1/4] 读取数据...
>>> [2/4] 清洗方程式并计算 free_energy...
>>> [3/4] 识别物种位置并进行矩阵编码...
数据处理完成！已保存至 processed_reaction_results.csv。


In [2]:
import csv
from py2neo import Graph, Node, Relationship


def build_neo4j_graph_from_encoded_csv():
    """
    基于编码CSV构建反应图谱

    编码规则：
    -1 反应物
     1 带*生成物
     2 不带*生成物
    """

    try:
        g = Graph('http://localhost:7474/', user='neo4j', password='mk910404445..')
        print("Neo4j数据库连接成功！开始构建反应网络...")
    except Exception as e:
        raise ConnectionError(f"Neo4j连接失败：{str(e)}")

    csv_file_path = "processed_reaction_results.csv"
    
    # 定义需要忽略的物种名称集合
    excluded_species = {"*H2O", "*"}

    with open(csv_file_path, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        header = None
        species_col_indexes = {}

        reaction_col_indexes = {
            "reaction_id": -1,
            "reactant_combination": -1,
            "response equation": -1,
            "free_energy": -1
        }

        for item in reader:
            if reader.line_num == 1:
                header = item
                for col_index, col_name in enumerate(header):
                    # 检查是否在排除名单中
                    if col_name in excluded_species:
                        continue
                        
                    # 识别物种列（非反应属性列）
                    if col_name not in reaction_col_indexes:
                        species_col_indexes[col_name] = col_index

                    if col_name in reaction_col_indexes:
                        reaction_col_indexes[col_name] = col_index
                continue

            # --- 提取反应基础信息 ---
            reaction_id = f"REACT_{reader.line_num - 1:03d}"
            reactant_combination = "未知"
            reaction_eqn = f"Reaction_{reader.line_num - 1}"
            free_energy_val = "未知"

            if reaction_col_indexes["reaction_id"] != -1:
                reaction_id = item[reaction_col_indexes["reaction_id"]]
            if reaction_col_indexes["reactant_combination"] != -1:
                reactant_combination = item[reaction_col_indexes["reactant_combination"]]
            if reaction_col_indexes["response equation"] != -1:
                reaction_eqn = item[reaction_col_indexes["response equation"]]
            if reaction_col_indexes["free_energy"] != -1:
                free_energy_val = item[reaction_col_indexes["free_energy"]]

            reactant_species_list = []
            product_star_list = []
            product_nonstar_list = []

            # --- 解析编码并过滤 ---
            for species_name, col_index in species_col_indexes.items():
                if col_index >= len(item): continue
                
                try:
                    species_value = int(item[col_index])
                except:
                    species_value = 0

                # 这里再次确认过滤（双重保险）
                if species_name in excluded_species:
                    continue

                if species_value == -1:
                    reactant_species_list.append(species_name)
                elif species_value == 1:
                    product_star_list.append(species_name)
                elif species_value == 2:
                    product_nonstar_list.append(species_name)

            # 如果过滤后该反应没有任何节点联系，可以选择跳过
            if not any([reactant_species_list, product_star_list, product_nonstar_list]):
                continue

            # ---------- 创建 Reaction 节点 ----------
            reaction_node = g.nodes.match("Reaction", reaction_id=reaction_id).first()
            if reaction_node is None:
                reaction_node = Node(
                    "Reaction",
                    reaction_id=reaction_id,
                    reactant_combination=reactant_combination,
                    reaction_equation=reaction_eqn
                )
                g.create(reaction_node)

            # ---------- 建立关系 (通用的处理函数或循环) ----------
            def create_species_rel(species_list, rel_type):
                for name in species_list:
                    # 获取或创建物种节点
                    s_node = g.nodes.match("Species", name=name).first()
                    if s_node is None:
                        s_node = Node("Species", name=name)
                        g.create(s_node)
                    
                    # 确定关系方向
                    if rel_type == "IS_REACTANT_OF":
                        rel = Relationship(s_node, rel_type, reaction_node, free_energy=free_energy_val)
                    else:
                        rel = Relationship(reaction_node, rel_type, s_node, free_energy=free_energy_val)
                    g.merge(rel)

            create_species_rel(reactant_species_list, "IS_REACTANT_OF")
            create_species_rel(product_star_list, "PRODUCES_STAR")
            create_species_rel(product_nonstar_list, "PRODUCES")

    print("=" * 50)
    print("✨ 反应图谱构建完成（已过滤 H2O, *H2O, *）！")

    print("开始执行查询：MATCH (n) RETURN n")
    
    # 使用 g.run() 执行原生的 Cypher 语句
    result = g.run("MATCH (n) RETURN n")
    

if __name__ == "__main__":
    try:
        build_neo4j_graph_from_encoded_csv()
    except Exception as e:
        print(f"程序运行出错：{str(e)}")

Neo4j数据库连接成功！开始构建反应网络...
✨ 反应图谱构建完成（已过滤 H2O, *H2O, *）！
开始执行查询：MATCH (n) RETURN n


In [1]:
import csv
import re
from py2neo import Graph

def find_chemical_pathways_bfs_limited(start_species, end_species, max_reaction_steps=3):
    """
    在 Neo4j 中搜索两点之间的化学反应路径
    采用 BFS with depth limit 思路，动态命名并导出带有简易路径的中英双语/英文表头 CSV
    """
    # 1. 连接数据库
    try:
        g = Graph('http://localhost:7474/', user='neo4j', password='mk910404445..')
    except Exception as e:
        print(f"❌ Neo4j连接失败：{str(e)}")
        return

    print(f"🔍 正在执行 BFS 穷举搜寻：从【{start_species}】到【{end_species}】...")
    print(f"⚙️ 深度限制 (Depth Limit): {max_reaction_steps} 步反应 (对应Neo4j中 {max_reaction_steps * 2} 条边)\n")

    # 2. 构造 Cypher 查询语句

    max_edges = max_reaction_steps * 2
    
    # 将 :Node 修改为 :Species，或者直接去掉标签使用 (start {name: $start_node})
    query = f"""
    MATCH p = (start:Species {{name: $start_node}})-[*1..{max_edges}]->(end:Species {{name: $end_node}})
    WHERE length(p) % 2 = 0
    RETURN p, 
           [r IN relationships(p) | r.free_energy] AS energies,
           [r IN relationships(p) | r.display_label] AS labels
    ORDER BY length(p) ASC
    """
    
    # 3. 执行查询
    results = g.run(query, start_node=start_species, end_node=end_species).data()
    
    if not results:
        print(f"⚠️ 在 {max_reaction_steps} 步反应深度限制内，未找到从 {start_species} 到 {end_species} 的路径。")
        return

    print(f"✅ 共找到 {len(results)} 条反应路径（已按反应步数从少到多排序）：\n")
    print("-" * 60)

    # 用来存储准备写入 CSV 的数据
    csv_data = []

    # 4. 解析并优美地打印路径，同时收集数据
    for index, record in enumerate(results, 1):
        path = record['p']
        nodes = path.nodes
        
        # 直接拿 Cypher 帮我们提取好的列表
        energies_list = record['energies']
        labels_list = record['labels']
        
        total_dg = 0.0
        has_invalid_dg = False
        
        # 复杂展示路径 (带反应和能量)
        path_display = f"({nodes[0]['name']})"
        # 纯净展示路径 (只有物种)
        simplified_path = f"({nodes[0]['name']})"
        
        actual_steps = 0
        for i in range(0, len(nodes) - 1, 2):
            reaction_node = nodes[i+1]          # 中间的蓝点
            product_node = nodes[i+2]           # 产物点
            
            reaction_id = reaction_node.get('reaction_id', 'Unknown')
            
            # 1. 优先从第一条边取自由能
            dg_str = energies_list[i]
            
            # 2. 如果第一条边是空/未知，去它的后半段（第二条边）取
            if dg_str is None or str(dg_str).strip() in ["", "未知", "None"]:
                dg_str = energies_list[i+1]
                
            # 3. 终极双重保险：如果底层实在没存好，从 display_label 字符串里“切”出数值
            if dg_str is None or str(dg_str).strip() in ["", "未知", "None"]:
                lbl = str(labels_list[i])
                if "ΔG:" in lbl:
                    dg_str = lbl.split("ΔG:")[-1].replace(")", "").strip()
                else:
                    dg_str = "0"
            
            # 计算总和
            try:
                val = float(str(dg_str).strip())
                total_dg += val
            except (ValueError, TypeError):
                has_invalid_dg = True 
                
            # 构建复杂路径：包含反应ID和ΔG
            path_display += f" ==[{reaction_id}, ΔG:{dg_str}]==> ({product_node['name']})"
            
            # 构建简易纯净路径：只用 -> 连接下一个物种
            simplified_path += f" -> ({product_node['name']})"
            
            actual_steps += 1
            
        # 终端打印展示 (保持丰富的输出信息)
        print(f"【路径 {index}】: 包含 {actual_steps} 步反应")
        print(f"🔀 完整路径: {path_display}")
        print(f"⏭️ 纯净路径: {simplified_path}")
        
        # 判断并记录最终的能量值
        if has_invalid_dg:
            dg_result = "Invalid/Missing Data"
            print(f"🔋 路径总 ΔG 估算: 无法计算(含未知数据)")
        else:
            dg_result = round(total_dg, 2)
            print(f"🔋 路径总 ΔG 估算: {dg_result:.2f}")
            
        print("-" * 60)

        # 把这行路径的数据存入列表
        csv_data.append([
            index,                  # Path_Index
            actual_steps,           # Reaction_Steps
            dg_result,              # Total_Free_Energy_dG
            path_display,           # Full_Reaction_Pathway
            simplified_path         # Simplified_Pathway (新增的纯净列)
        ])

    # 5. 动态生成文件名并写入 CSV 文件
    # 过滤掉系统文件名不支持的星号等特殊字符
    safe_end_species = re.sub(r'[\\/*?:"<>|]', "", end_species)
    output_filename = f"{safe_end_species}_pathways.csv"

    try:
        with open(output_filename, mode='w', encoding='utf-8-sig', newline='') as f:
            writer = csv.writer(f)
            # 写入全英文表头
            writer.writerow([
                'Path_Index', 
                'Reaction_Steps', 
                'Total_Free_Energy_dG', 
                'Full_Reaction_Pathway', 
                'Simplified_Pathway'
            ])
            # 写入所有数据
            writer.writerows(csv_data)
        print(f"🎉 完美收工！所有路径已成功保存至文件：【{output_filename}】")
    except Exception as e:
        print(f"❌ 写入 CSV 文件时发生错误：{str(e)}")

# ==========================================
# 主程序
# ==========================================
if __name__ == "__main__":
    START_SPECIES = "CO2"   
    END_SPECIES = "CH2CH2"        
    MAX_STEPS = 20 
    
    # 无需再手动传 output_csv，代码会在内部使用 END_SPECIES 自动生成文件名
    find_chemical_pathways_bfs_limited(
        start_species=START_SPECIES, 
        end_species=END_SPECIES, 
        max_reaction_steps=MAX_STEPS
    )

🔍 正在执行 BFS 穷举搜寻：从【CO2】到【CH2CH2】...
⚙️ 深度限制 (Depth Limit): 20 步反应 (对应Neo4j中 40 条边)

✅ 共找到 99 条反应路径（已按反应步数从少到多排序）：

------------------------------------------------------------
【路径 1】: 包含 11 步反应
🔀 完整路径: (CO2) ==[REACT_001, ΔG:0.0264]==> (*CO2) ==[REACT_002, ΔG:2.0926]==> (*COOH) ==[REACT_003, ΔG:-1.5627]==> (*CO) ==[REACT_061, ΔG:1.4768]==> (*COCHO) ==[REACT_014, ΔG:0.9108]==> (*COHCHO) ==[REACT_008, ΔG:-0.6508]==> (*COHCH2O) ==[REACT_009, ΔG:-0.6015]==> (*COHCH2OH) ==[REACT_010, ΔG:-0.4852]==> (*CHOHCH2OH) ==[REACT_011, ΔG:-1.1691]==> (CH2OHCH2OH) ==[REACT_012, ΔG:0.4672]==> (*CH2OHCH2) ==[REACT_013, ΔG:-1.4043]==> (CH2CH2)
⏭️ 纯净路径: (CO2) -> (*CO2) -> (*COOH) -> (*CO) -> (*COCHO) -> (*COHCHO) -> (*COHCH2O) -> (*COHCH2OH) -> (*CHOHCH2OH) -> (CH2OHCH2OH) -> (*CH2OHCH2) -> (CH2CH2)
🔋 路径总 ΔG 估算: -0.90
------------------------------------------------------------
【路径 2】: 包含 11 步反应
🔀 完整路径: (CO2) ==[REACT_001, ΔG:0.0264]==> (*CO2) ==[REACT_002, ΔG:2.0926]==> (*COOH) ==[REACT_003, ΔG:-1.5627]==>

In [6]:
import csv
import re
from py2neo import Graph

def find_chemical_pathways_bfs_limited(start_species, end_species, max_reaction_steps=3):
    """
    在 Neo4j 中搜索两点之间的化学反应路径
    采用 BFS 思路，导出带有简易路径的 CSV
    新增功能：计算并输出 能量跨度 (Energetic Span, dE)
    """
    try:
        g = Graph('http://localhost:7474/', user='neo4j', password='mk910404445..')
    except Exception as e:
        print(f"❌ Neo4j连接失败：{str(e)}")
        return

    print(f"🔍 正在执行 BFS 穷举搜寻：从【{start_species}】到【{end_species}】...")
    print(f"⚙️ 深度限制 (Depth Limit): {max_reaction_steps} 步反应\n")

    max_edges = max_reaction_steps * 2
    
    query = f"""
    MATCH p = (start:Species {{name: $start_node}})-[*1..{max_edges}]->(end:Species {{name: $end_node}})
    WHERE length(p) % 2 = 0
    RETURN p, 
           [r IN relationships(p) | r.free_energy] AS energies,
           [r IN relationships(p) | r.display_label] AS labels
    ORDER BY length(p) ASC
    """
    
    results = g.run(query, start_node=start_species, end_node=end_species).data()
    
    if not results:
        print(f"⚠️ 在 {max_reaction_steps} 步反应深度限制内，未找到路径。")
        return

    print(f"✅ 共找到 {len(results)} 条反应路径（已按反应步数从少到多排序）：\n")
    print("-" * 60)

    csv_data = []

    for index, record in enumerate(results, 1):
        path = record['p']
        nodes = path.nodes
        
        energies_list = record['energies']
        labels_list = record['labels']
        
        total_dg = 0.0
        total_positive_dg = 0.0
        
        # [新增] 记录路径上的相对能量曲线，初始能量设为 0.0
        cumulative_energies = [0.0] 
        current_g = 0.0
        
        has_invalid_dg = False
        
        path_display = f"({nodes[0]['name']})"
        simplified_path = f"({nodes[0]['name']})"
        
        actual_steps = 0
        for i in range(0, len(nodes) - 1, 2):
            reaction_node = nodes[i+1]
            product_node = nodes[i+2]
            
            reaction_id = reaction_node.get('reaction_id', 'Unknown')
            
            dg_str = energies_list[i]
            if dg_str is None or str(dg_str).strip() in ["", "未知", "None"]:
                dg_str = energies_list[i+1]
                
            if dg_str is None or str(dg_str).strip() in ["", "未知", "None"]:
                lbl = str(labels_list[i])
                if "ΔG:" in lbl:
                    dg_str = lbl.split("ΔG:")[-1].replace(")", "").strip()
                else:
                    dg_str = "0"
            
            try:
                val = float(str(dg_str).strip())
                total_dg += val
                if val > 0:
                    total_positive_dg += val
                    
                # [新增] 计算当前步骤结束后的累积相对能量，并记录
                current_g += val
                cumulative_energies.append(current_g)
                
            except (ValueError, TypeError):
                has_invalid_dg = True 
                
            path_display += f" ==[{reaction_id}, ΔG:{dg_str}]==> ({product_node['name']})"
            simplified_path += f" -> ({product_node['name']})"
            
            actual_steps += 1
            
        # [新增] 计算能量跨度 (Energetic Span)
        # 算法: 寻找路径中 max(G_j - G_i)，其中 j >= i
        if not has_invalid_dg:
            energetic_span = 0.0
            for i in range(len(cumulative_energies)):
                for j in range(i, len(cumulative_energies)):
                    span = cumulative_energies[j] - cumulative_energies[i]
                    if span > energetic_span:
                        energetic_span = span
                        
            span_result = round(energetic_span, 2)
            dg_result = round(total_dg, 2)
            pos_dg_result = round(total_positive_dg, 2)
        else:
            span_result = "Invalid/Missing Data"
            dg_result = "Invalid/Missing Data"
            pos_dg_result = "Invalid/Missing Data"
            
        print(f"【路径 {index}】: 包含 {actual_steps} 步反应")
        print(f"🔀 完整路径: {path_display}")
        print(f"⏭️ 纯净路径: {simplified_path}")
        print(f"🔋 路径总 ΔG 估算: {dg_result}")
        print(f"⛰️  正向 ΔG 能垒和: {pos_dg_result}")
        print(f"🌉 能量跨度 (Energetic Span): {span_result}")  # 打印能量跨度
        print("-" * 60)

        csv_data.append([
            index,                  
            actual_steps,           
            dg_result,              
            pos_dg_result,          
            span_result,            # 将能量跨度存入列表
            path_display,           
            simplified_path         
        ])

    safe_end_species = re.sub(r'[\\/*?:"<>|]', "", end_species)
    output_filename = f"{safe_end_species}_pathways.csv"

    try:
        with open(output_filename, mode='w', encoding='utf-8-sig', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([
                'Path_Index', 
                'Reaction_Steps', 
                'Total_Free_Energy_dG', 
                'Sum_Positive_dG_Barrier',
                'Energetic_Span_dE',    # 新增表头
                'Full_Reaction_Pathway', 
                'Simplified_Pathway'
            ])
            writer.writerows(csv_data)
        print(f"🎉 完美收工！所有路径已成功保存至文件：【{output_filename}】")
    except Exception as e:
        print(f"❌ 写入 CSV 文件时发生错误：{str(e)}")


if __name__ == "__main__":
    START_SPECIES = "CO2"   
    END_SPECIES = "CH2CH2"        
    MAX_STEPS = 20 
    
    find_chemical_pathways_bfs_limited(
        start_species=START_SPECIES, 
        end_species=END_SPECIES, 
        max_reaction_steps=MAX_STEPS
    )

🔍 正在执行 BFS 穷举搜寻：从【CO2】到【CH2CH2】...
⚙️ 深度限制 (Depth Limit): 20 步反应

✅ 共找到 99 条反应路径（已按反应步数从少到多排序）：

------------------------------------------------------------
【路径 1】: 包含 11 步反应
🔀 完整路径: (CO2) ==[REACT_001, ΔG:0.0264]==> (*CO2) ==[REACT_002, ΔG:2.0926]==> (*COOH) ==[REACT_003, ΔG:-1.5627]==> (*CO) ==[REACT_061, ΔG:1.4768]==> (*COCHO) ==[REACT_014, ΔG:0.9108]==> (*COHCHO) ==[REACT_008, ΔG:-0.6508]==> (*COHCH2O) ==[REACT_009, ΔG:-0.6015]==> (*COHCH2OH) ==[REACT_010, ΔG:-0.4852]==> (*CHOHCH2OH) ==[REACT_011, ΔG:-1.1691]==> (CH2OHCH2OH) ==[REACT_012, ΔG:0.4672]==> (*CH2OHCH2) ==[REACT_013, ΔG:-1.4043]==> (CH2CH2)
⏭️ 纯净路径: (CO2) -> (*CO2) -> (*COOH) -> (*CO) -> (*COCHO) -> (*COHCHO) -> (*COHCH2O) -> (*COHCH2OH) -> (*CHOHCH2OH) -> (CH2OHCH2OH) -> (*CH2OHCH2) -> (CH2CH2)
🔋 路径总 ΔG 估算: -0.9
⛰️  正向 ΔG 能垒和: 4.97
🌉 能量跨度 (Energetic Span): 2.94
------------------------------------------------------------
【路径 2】: 包含 11 步反应
🔀 完整路径: (CO2) ==[REACT_001, ΔG:0.0264]==> (*CO2) ==[REACT_002, ΔG:2.0926]==> (*CO